# ListSmart: build `listsmart-writer` on Google Colab

**What this notebook does:** it takes our fine-tuned LoRA from Tinker, bakes it into Qwen3-8B,
and gives you **one file** (`listsmart-writer-Q4_K_M.gguf`, about 5 GB) plus a small `Modelfile`.
You then copy those two files to your Mac and run `ollama create`.

**Why Colab:** the export needs about **23-27 GB of free disk** for a short time (an 8.7 GB base model,
a 16 GB merged file, then the 5 GB final file). The 8 GB M1 Mac only has ~12 GB free.
Colab gives you a Linux machine with ~100 GB of disk and ~12 GB of RAM, for free.

**Before you start**
- You have already run `python train.py` on the Mac, so you have a `tinker://...` path
  (in `finetune/results/external_weights.txt` or `finetune/results/finetuned_model.txt`).
- You have your `TINKER_API_KEY`.
- A normal **CPU runtime is enough** (no GPU needed). Menu: *Runtime -> Change runtime type -> CPU*.
- Time: roughly 30-60 minutes, mostly downloading and writing big files.

Run the cells **from top to bottom**. If Colab disconnects, just run them again: every export
step skips itself when its output file already exists (as long as the Colab machine was not reset).

## Step 1: get the code

You only need the `finetune/` and `backend/app/` folders. Pick **one** of the two options below.

### Option A: `git clone` (easiest if the repo is on GitHub)

Put your repo address in `REPO_URL`. For a **private** repo, use a GitHub token in the address:
`https://<your-token>@github.com/<you>/ListSmart.git` (make the token read-only, and do not share the notebook with it inside).

In [ ]:
import os, subprocess

# Change this to your repo, e.g. "https://github.com/<your-github-name>/ListSmart.git"
REPO_URL = "https://github.com/<your-github-name>/ListSmart.git"
USE_GIT = True   # set to False if you use Option B (upload a zip) instead

ROOT = "/content/ListSmart"
if USE_GIT:
    if "<your-github-name>" in REPO_URL:
        print("Please put your real repo address in REPO_URL first (or set USE_GIT = False and use Option B).")
    elif os.path.exists(ROOT):
        print("Already cloned:", ROOT)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], check=True)
        print("Cloned into", ROOT)

### Option B: upload a zip from your Mac

On the Mac, make a small zip with only the folders we need (this leaves out the big venvs and data):

```bash
cd ~/Desktop
zip -r listsmart_export.zip ListSmart/finetune ListSmart/backend/app \
  -x '*/.venv/*' '*/export/*' '*/__pycache__/*' '*/data/train.jsonl' '*/data/val.jsonl' '*/.env'
```

Then run the cell below and pick `listsmart_export.zip`. (Skip this cell if you used Option A.)

In [ ]:
import os, zipfile
from google.colab import files

# Opens a file picker; choose listsmart_export.zip
uploaded = files.upload()
for name in uploaded:
    with zipfile.ZipFile(name) as z:
        z.extractall("/content")
    print("Unzipped", name)
ROOT = "/content/ListSmart"

In [ ]:
# Check that the code is in place
import os
ROOT = "/content/ListSmart"
FINETUNE = os.path.join(ROOT, "finetune")
EXPORT_DIR = os.path.join(FINETUNE, "export")
for path in ["finetune/export_to_ollama.py", "finetune/requirements.txt", "finetune/common.py",
             "backend/app/prompts.py", "backend/app/chat_template.py"]:
    full = os.path.join(ROOT, path)
    print(("OK      " if os.path.exists(full) else "MISSING ") + path)

## Step 2: install the Python packages and build the llama.cpp tools

- `finetune/requirements.txt` gives us the Tinker SDK + cookbook (they also bring `torch`,
  `transformers`, `safetensors` and `huggingface_hub`, which Colab mostly has already).
- llama.cpp gives us the converter scripts and two small programs:
  `llama-export-lora` (merges the LoRA into the base model) and `llama-quantize` (makes the 5 GB Q4_K_M file).
  We build only those two, which takes about 5-10 minutes on Colab.

In [ ]:
%cd {FINETUNE}
!pip install -q -r requirements.txt
# Small extras the llama.cpp converter scripts use (safe to install again)
!pip install -q sentencepiece protobuf

In [ ]:
import os, shutil, subprocess

LLAMA_DIR = os.path.join(EXPORT_DIR, "llama.cpp")
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Get the llama.cpp source (the export script would also do this, we just do it here so you can see it)
if not os.path.exists(os.path.join(LLAMA_DIR, "convert_lora_to_gguf.py")):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA_DIR], check=True)

# 2. Make sure cmake exists (Colab normally has it)
if not shutil.which("cmake"):
    subprocess.run(["apt-get", "install", "-y", "-qq", "cmake"], check=True)

# 3. Build only the two tools we need. LLAMA_CURL=OFF: we do not need llama.cpp's own downloader.
build = os.path.join(LLAMA_DIR, "build")
subprocess.run(["cmake", "-S", LLAMA_DIR, "-B", build, "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"], check=True)
subprocess.run(["cmake", "--build", build, "--config", "Release", "-j", str(os.cpu_count() or 2),
                "--target", "llama-export-lora", "llama-quantize"], check=True)

for tool in ["llama-export-lora", "llama-quantize"]:
    path = os.path.join(build, "bin", tool)
    print(("OK      " if os.path.exists(path) else "MISSING ") + path)

## Step 3: your Tinker API key

The key is only kept in this Colab session's memory (as an environment variable). It is not saved
in the notebook or written to any file.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("TINKER_API_KEY"):
    os.environ["TINKER_API_KEY"] = getpass("Paste your TINKER_API_KEY (hidden): ").strip()
print("TINKER_API_KEY is set" if os.environ["TINKER_API_KEY"] else "TINKER_API_KEY is EMPTY")

## Step 4: which fine-tuned checkpoint?

Paste the `tinker://...` path, **or** leave `TINKER_PATH` empty and the cell reads it from the repo:
first `finetune/results/external_weights.txt` (the newer, downloadable copy, preferred),
then `finetune/results/finetuned_model.txt`.

(Those files only exist here if you committed/zipped them after training. If not, open the file on
your Mac and paste its content below.)

In [ ]:
import os

TINKER_PATH = ""   # e.g. "tinker://<run-id>/external_weights/listsmart-writer"  (leave empty to read it from results/)

if not TINKER_PATH:
    for name in ["external_weights.txt", "finetuned_model.txt"]:
        f = os.path.join(FINETUNE, "results", name)
        if os.path.exists(f) and open(f).read().strip():
            TINKER_PATH = open(f).read().strip()
            print("Read from results/" + name)
            break

if TINKER_PATH.startswith("tinker://"):
    print("Using:", TINKER_PATH)
else:
    print("No tinker:// path found. Paste it into TINKER_PATH above and run this cell again.")

## Step 5: run the export

We run `finetune/export_to_ollama.py` (the same script as on the Mac) with:

- `--tinker-path ...` : the checkpoint from Step 4.
- `--cleanup` : delete each big in-between file as soon as it is used (keeps disk use lower).
- `--skip create` : **do not** run `ollama create`, because Colab has no Ollama. You do that on the Mac.
- Default `--mode adapter`: LoRA -> GGUF, merge into the official `Qwen/Qwen3-8B-GGUF` Q8_0 file, then make Q4_K_M.
  We do **not** use `--base-gguf ollama` here (that reuses a model from a local Ollama, which Colab does not have).

The steps are: setup, download, peft, lora_gguf, base_gguf, merge_gguf, quantize, modelfile.

First a **dry run**: it only prints the plan and the disk check, nothing is downloaded.

In [ ]:
%cd {FINETUNE}
# How much disk this Colab machine has free (the export needs ~27 GB at its peak)
!df -h /content | tail -1
!python export_to_ollama.py --dry-run --cleanup --skip create --tinker-path "{TINKER_PATH}"

Now the **real run** (about 30-60 minutes). You will see each step's progress.
If it stops (for example Colab disconnects), run this cell again: finished steps are skipped.

In [ ]:
%cd {FINETUNE}
!python export_to_ollama.py --cleanup --skip create --tinker-path "{TINKER_PATH}"

**If a step fails**
- `ModuleNotFoundError: No module named 'xyz'` in the `lora_gguf` step: run `!pip install -q xyz` in a new cell, then run the export cell again.
- A Tinker error in the `download` step: check the key (Step 3) and the `tinker://` path (Step 4).
  Paths saved by `train.py`'s `save_weights_external` (in `external_weights.txt`) are the most reliable.
- "not enough free disk": you are not on a normal Colab machine; *Runtime -> Disconnect and delete runtime* and start again.

In [ ]:
# Check the two files we need
import os
GGUF = os.path.join(EXPORT_DIR, "listsmart-writer-Q4_K_M.gguf")
MODELFILE = os.path.join(EXPORT_DIR, "Modelfile")
for path in [GGUF, MODELFILE]:
    if os.path.exists(path):
        print(f"OK  {path}  ({os.path.getsize(path) / 1e9:.2f} GB)")
    else:
        print("MISSING", path, "- look at the export output above for the error")

# The Modelfile must point to the .gguf next to it (a relative ./ path), so it works on the Mac too
print(open(MODELFILE).read() if os.path.exists(MODELFILE) else "")

## Step 6: get the files to your Mac

**Google Drive is the better way** for a 5 GB file: the copy runs inside Google's network (fast),
and you can then download it from drive.google.com or the Google Drive app on the Mac.
You need about **5 GB free in your Google Drive** (the free plan has 15 GB).

The direct browser download (second cell below) also works, but a 5 GB download from Colab is slow and often fails.

In [ ]:
# Option 1 (recommended): copy to Google Drive -> MyDrive/ListSmart-export/
import os, shutil
from google.colab import drive

drive.mount("/content/drive")
target = "/content/drive/MyDrive/ListSmart-export"
os.makedirs(target, exist_ok=True)
for path in [MODELFILE, GGUF]:
    print("copying", os.path.basename(path), "...")
    shutil.copy(path, target)
print("Done. On your Mac, download both files from Google Drive -> ListSmart-export")

In [ ]:
# Option 2: download straight to the Mac's Downloads folder (slow for 5 GB, may fail; try Drive first)
from google.colab import files
files.download(MODELFILE)
files.download(GGUF)

## Step 7: finish on your Mac

Disk tip: the Mac needs about **10 GB free for a moment** (the 5 GB `.gguf` + Ollama's own 5 GB copy).
After `ollama create` you can delete the `.gguf`. Run `python3 scripts/doctor.py` to see your free space.

Open Terminal on the Mac:

```bash
cd ~/Desktop/ListSmart
mkdir -p finetune/export

# 1. Put the two files in finetune/export/ (change ~/Downloads if you saved them somewhere else)
mv ~/Downloads/listsmart-writer-Q4_K_M.gguf ~/Downloads/Modelfile finetune/export/
#    (if the browser renamed it to Modelfile.txt:  mv ~/Downloads/Modelfile.txt finetune/export/Modelfile)

# 2. Check the FROM line. It must be:  FROM ./listsmart-writer-Q4_K_M.gguf
grep '^FROM' finetune/export/Modelfile
#    If it shows something else (for example a /content/... path), fix it with:
sed -i '' 's|^FROM .*|FROM ./listsmart-writer-Q4_K_M.gguf|' finetune/export/Modelfile

# 3. Start Ollama (open the Ollama app, or run `ollama serve` in another Terminal tab), then create the model
ollama create listsmart-writer -f finetune/export/Modelfile
ollama list            # listsmart-writer:latest should be in the list

# 4. Smoke test: it should answer with a listing-style JSON (it was trained to do that). Type /bye to quit.
ollama run listsmart-writer "Product: maroon cotton kurti, block print, 3/4 sleeves. Platform: meesho. Write the listing JSON."

# 5. Ollama keeps its own copy, so free 5 GB by deleting the .gguf
rm finetune/export/listsmart-writer-Q4_K_M.gguf

# 6. Check everything again
python3 scripts/doctor.py
```

Then:
- Start the app (`WRITER_BACKEND=ollama` and `OLLAMA_WRITER_MODEL=listsmart-writer` are the defaults in `.env.example`).
- Optional: measure it on the laptop with
  `cd finetune && .venv/bin/python evaluate.py --backend ollama --limit 20 --no-judge`
  (needs `ollama pull qwen3:8b` too, ~5.2 GB, for the base-model comparison).

**Done on Colab?** Delete the files from Google Drive later if you need the space, and
*Runtime -> Disconnect and delete runtime* to free the Colab machine.